In [1]:
import os
import zipfile
import pyarrow.parquet as pq
import pandas as pd

folder = r"F:\sanazi\python\data science\code\tamrin\proje\Tennis Schema\Tennis Schema\tennis_data"

data = {}

for file in os.listdir(folder): 
    if file.endswith(".zip"): 
        zip_path = os.path.join(folder, file) 

        with zipfile.ZipFile(zip_path) as z: 
            for parquet_file in z.namelist():
                if parquet_file.endswith(".parquet"): 

                    with z.open(parquet_file) as f: 
                        df = pq.read_table(f).to_pandas()
                    table_name =os.path.basename(os.path.dirname(parquet_file))[4:-8] + os.path.basename(parquet_file)[:-17] 

                    if table_name not in data:
                        data[table_name] = [] 
                    data[table_name].append(df) 


for table_name in data:
    data[table_name] = pd.concat(data[table_name], ignore_index=True)

print(data.keys())

C:\Users\sepehr\AppData\Local\Temp\ipykernel_23392\661795369.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data[table_name] = pd.concat(data[table_name], ignore_index=True)


dict_keys(['matchaway_team', 'matchaway_team_score', 'matchevent', 'matchhome_team', 'matchhome_team_score', 'matchround', 'matchseason', 'matchtime', 'matchtournament', 'matchvenue', 'oddsodds', 'point_by_pointpbp', 'statisticsstatistics', 'tennis_powerpower', 'votesvotes'])


In [3]:
print(data["matchhome_team_score"].columns)

Index(['match_id', 'current_score', 'display_score', 'period_1', 'period_2',
       'period_3', 'period_4', 'period_5', 'period_1_tie_break',
       'period_2_tie_break', 'period_3_tie_break', 'period_4_tie_break',
       'period_5_tie_break', 'normal_time'],
      dtype='object')


In [4]:
print(data["matchaway_team_score"].columns)

Index(['match_id', 'current_score', 'display_score', 'period_1', 'period_2',
       'period_3', 'period_4', 'period_5', 'period_1_tie_break',
       'period_2_tie_break', 'period_3_tie_break', 'period_4_tie_break',
       'period_5_tie_break', 'normal_time'],
      dtype='object')


In [5]:
import pandas as pd
import numpy as np

home_score = data["matchhome_team_score"]
away_score = data["matchaway_team_score"]

event = data["matchevent"][["match_id", "winner_code"]]


df = home_score.merge(
    away_score,
    on="match_id",
    suffixes=("_home", "_away")
)


df = df.merge(event, on="match_id")


df = df.dropna(subset=["period_1_home", "period_1_away"])


df["first_set_winner"] = np.where(
    df["period_1_home"] > df["period_1_away"],
    1,
    2
)


df["same_winner"] = df["first_set_winner"] == df["winner_code"]


success_rate = df["same_winner"].mean() * 100

print(f"Winning percentage of first-set winner: {success_rate:.2f}%")

print("\nDistribution:")
print(df["same_winner"].value_counts())

Winning percentage of first-set winner: 82.03%

Distribution:
same_winner
True     122944
False     26933
Name: count, dtype: int64
